In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sunilthite/llm-detect-ai-generated-text-dataset")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/llm-detect-ai-generated-text-dataset


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
import pandas as pd

from tqdm import tqdm
from transformers import T5ForConditionalGeneration, T5Tokenizer, AutoModelForCausalLM, AutoTokenizer
from typing import List

In [3]:
dataset = pd.read_csv("/kaggle/input/llm-detect-ai-generated-text-dataset/Training_Essay_Data.csv")
dataset.shape

(29145, 2)

In [4]:
# We will create a smaller dataset consisting of 40% LLM generated and 60% human generated text
llm_dataset = dataset[dataset['generated'] == 1]
human_dataset = dataset[dataset['generated'] == 0]
print(llm_dataset.shape, human_dataset.shape)

llm_dataset = llm_dataset[:int((llm_dataset.shape[0])*0.05)]
human_dataset = human_dataset[:int((human_dataset.shape[0])*0.03)]
print(llm_dataset.shape, human_dataset.shape)

smaller_dataset = pd.concat([llm_dataset, human_dataset])
smaller_dataset = smaller_dataset.sample(frac=1)
smaller_dataset.shape

(11637, 2) (17508, 2)
(581, 2) (525, 2)


(1106, 2)

In [5]:
print(smaller_dataset['generated'].value_counts())

generated
1    581
0    525
Name: count, dtype: int64


In [6]:
smaller_dataset.head(30)

,text,generated
462,"[Your Name]\n[Your Address]\n[City, State, ZIP...",1
1232,I think that they should not change the Electo...,0
405,"[Your Name]\n[Your Address]\n[City, State, Zip...",1
899,"Dear Senator, Good day, I am writing this lett...",0
133,Limiting car usage can provide numerous advant...,1
35,Transforming Urban Living for a Sustainable ...,1
914,"It is often said that ""change is good."" This s...",0
7,Pioneering Sustainable Urban Living In an a...,1
1057,To my fellow citizens all across the world I t...,0
57,Advantages of Limiting Car Usage\n\nLimiting c...,1


In [7]:
class PerturbationGenerator:
    def __init__(self, model_name: str = "t5-large"):
        self.tokenizer = T5Tokenizer.from_pretrained(model_name)
        self.tokenizer.model_max_length = 512
        self.model = T5ForConditionalGeneration.from_pretrained(model_name, torch_dtype=torch.float16)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)

    def perturb(self, text: str, num_samples: int = 9) -> List[str]:
        inputs = self.tokenizer(
            f"paraphrase: {text}", return_tensors="pt", max_length=512, truncation=True
        ).to(self.device)

        outputs = self.model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            num_return_sequences=num_samples,
            do_sample=True,
            max_length=512,
        )
        
        return [self.tokenizer.decode(out, skip_special_tokens=True) for out in outputs]



class SourceModel:
    def __init__(self, model_name="gpt2"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()

        
    def get_log_prob(self, texts: List[str]) -> torch.Tensor:
        log_probs = []
        with torch.no_grad():
            for text in texts:
                inputs = self.tokenizer(text, return_tensors="pt", truncation=True).to(self.device)
                
                # Handle empty input
                if inputs["input_ids"].shape[1] == 0:
                    log_probs.append(0.0)
                    continue
                
                # Handle single-token edge case
                num_tokens = inputs["input_ids"].shape[1]
                if num_tokens == 1:
                    log_probs.append(0.0)
                    continue
                
                outputs = self.model(**inputs, labels=inputs["input_ids"])
                logits = outputs.logits
                labels = inputs["input_ids"]
                
                log_probs_seq = torch.log_softmax(logits, dim=-1)[:, :-1, :]
                log_probs_text = torch.gather(
                    log_probs_seq, 
                    dim=2, 
                    index=labels[:, 1:, None]
                ).squeeze(-1).sum(dim=1)

                normalized_log_prob = log_probs_text.item() / (num_tokens - 1)  
                log_probs.append(normalized_log_prob)
            
        # print(log_probs)    
        return torch.tensor(log_probs)




class ScoreCalculator:
    def __init__(self):
        self.perturb_model = PerturbationGenerator(model_name='t5-large')
        self.source_model = SourceModel(model_name='gpt2')

    def score(self, candidate_text):
        perturbations = self.perturb_model.perturb(text=candidate_text, num_samples=100)     #   Increase for better accuracy
        candidate_text_log_prob = self.source_model.get_log_prob([candidate_text])
        perturbations_log_prob = self.source_model.get_log_prob(perturbations)

        detection_score = candidate_text_log_prob - perturbations_log_prob.mean()
        return detection_score

In [ ]:
if __name__ == '__main__':
    sc = ScoreCalculator()
    sentences = list(smaller_dataset['text'])
    pred_scores = []
    for num, sentence in enumerate(sentences, 1):
        pred_scores.append(sc.score(candidate_text=sentence).item())
        print(f"Detection Score for sentence {num} is: {pred_scores[num-1]:0.4f}")
        torch.cuda.empty_cache()

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


model.safetensors:   0%|          | 0.00/2.95G [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Detection Score for sentence 1 is: 1.8333
Detection Score for sentence 2 is: 0.6303
Detection Score for sentence 3 is: 1.6478
Detection Score for sentence 4 is: 1.2115
Detection Score for sentence 5 is: 1.5941
Detection Score for sentence 6 is: 1.9716
Detection Score for sentence 7 is: 1.4221
Detection Score for sentence 8 is: 0.6507
Detection Score for sentence 9 is: 1.4543
Detection Score for sentence 10 is: 2.5773
Detection Score for sentence 11 is: 1.6953
Detection Score for sentence 12 is: 1.9255
Detection Score for sentence 13 is: 0.8199
Detection Score for sentence 14 is: 0.5566
Detection Score for sentence 15 is: 1.4018
Detection Score for sentence 16 is: 1.0676
Detection Score for sentence 17 is: 1.0514
Detection Score for sentence 18 is: 2.2073
Detection Score for sentence 19 is: 1.8628
Detection Score for sentence 20 is: 0.6120
Detection Score for sentence 21 is: 1.1896
Detection Score for sentence 22 is: 2.2916
Detection Score for sentence 23 is: 1.9360
Detection Score for 

### Set the threshold as 1.4

If detection score > 1.4, LLM generated else Human-generated